# ASR Fine-tuning: PhoWhisper-base trên VITOSA


| Siêu tham số | Giá trị |
|---|---|
| Model | `vinai/PhoWhisper-base` |
| Dataset | `UIT-ViToSA/ViToSA-1.0` |
| Epochs | 10 |
| Batch size | 8 |
| Optimizer | AdamW |
| Learning rate | 5e-5 |
| Warmup ratio | 0.1 |
| Weight decay | 0.05 |
| Metric | WER (toxic / non-toxic / all) |


## 1. Cài đặt thư viện

In [1]:
import os
import random
import numpy as np
import torch

def set_seed(seed_value=42):
    """Cố định seed cho tất cả các thư viện để đảm bảo kết quả đồng nhất."""
    # 1. Đặt seed cho Python built-in và môi trường
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    # 2. Đặt seed cho NumPy
    np.random.seed(seed_value)
    
    # 3. Đặt seed cho PyTorch (CPU & All GPUs)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)  # Nếu dùng multi-GPU
        
        # 4. Đảm bảo cấu hình thuật toán của CUDA là cố định
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Gọi hàm để kích hoạt seed cố định (thường dùng số 42 hoặc số bạn thích)
GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)
print(f"✅ Đã thiết lập seed cố định: {GLOBAL_SEED}")

✅ Đã thiết lập seed cố định: 42


In [2]:
!pip install -q transformers datasets accelerate huggingface_hub
!pip install -q jiwer soundfile librosa audioread
!pip install -q evaluate
!pip install -q accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


## 2. Đăng nhập Hugging Face

In [3]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Lấy HF token từ Kaggle Secrets (thêm secret tên 'HF_TOKEN' trong Kaggle)
try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('Đăng nhập Hugging Face thành công!')
except Exception as e:
    print(f'Không tìm thấy secret, thử nhập thủ công: {e}')
    # fallback: login(token='hf_xxx...')

Đăng nhập Hugging Face thành công!


## 3. Tải và khám phá dataset VITOSA

In [4]:
from datasets import load_dataset, DatasetDict, Audio

print('Đang tải dataset UIT-ViToSA/ViToSA-1.0 ...')
raw_dataset = load_dataset('UIT-ViToSA/ViToSA-1.0', trust_remote_code=True)
print(raw_dataset)
print('\n--- Các cột dữ liệu ---')
print(raw_dataset['train'].features)
print('\n--- Ví dụ mẫu đầu tiên ---')
# print(raw_dataset['train'][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'UIT-ViToSA/ViToSA-1.0' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Đang tải dataset UIT-ViToSA/ViToSA-1.0 ...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00012.parquet:   0%|          | 0.00/921M [00:00<?, ?B/s]

data/train-00001-of-00012.parquet:   0%|          | 0.00/936M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/8641 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2161 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['file_name', 'audio', 'transcript', 'toxicity', 'split', 'label_binary_token_level', 'BIO_Tag', 'toxic_spans_text', 'toxic_spans_char_offsets'],
        num_rows: 8641
    })
    validation: Dataset({
        features: ['file_name', 'audio', 'transcript', 'toxicity', 'split', 'label_binary_token_level', 'BIO_Tag', 'toxic_spans_text', 'toxic_spans_char_offsets'],
        num_rows: 2161
    })
    test: Dataset({
        features: ['file_name', 'audio', 'transcript', 'toxicity', 'split', 'label_binary_token_level', 'BIO_Tag', 'toxic_spans_text', 'toxic_spans_char_offsets'],
        num_rows: 2000
    })
})

--- Các cột dữ liệu ---
{'file_name': Value('string'), 'audio': Audio(sampling_rate=None, decode=False, num_channels=None, stream_index=None), 'transcript': Value('string'), 'toxicity': Value('int64'), 'split': Value('string'), 'label_binary_token_level': Value('string'), 'BIO_Tag': Value('string'), 'toxic_spans_text': Value('strin

In [5]:
# Kiểm tra số lượng mẫu mỗi split
for split in raw_dataset:
    n = len(raw_dataset[split])
    print(f'Split [{split:12s}]: {n:,} mẫu')

# Xác định tên cột (thích nghi theo cấu trúc dataset thực tế)
sample = raw_dataset['train'][0]
print('\n--- Tên các cột ---', list(sample.keys()))

Split [train       ]: 8,641 mẫu
Split [validation  ]: 2,161 mẫu
Split [test        ]: 2,000 mẫu

--- Tên các cột --- ['file_name', 'audio', 'transcript', 'toxicity', 'split', 'label_binary_token_level', 'BIO_Tag', 'toxic_spans_text', 'toxic_spans_char_offsets']


In [6]:
# ---- Điều chỉnh tên cột nếu cần ----

AUDIO_COL  = 'audio'       # cột chứa audio
TEXT_COL   = 'transcript'  # cột chứa transcript text
LABEL_COL  = 'toxicity'       # cột nhãn toxic/non-toxic (nếu có)

# Cast cột audio về 16kHz mono
raw_dataset = raw_dataset.cast_column(AUDIO_COL, Audio(sampling_rate=16000))
print('Cast audio sang 16kHz thành công')

Cast audio sang 16kHz thành công


In [7]:
# Thống kê nhãn toxic / non-toxic trên test set
from collections import Counter

test_labels = raw_dataset['test'][LABEL_COL]
cnt = Counter(test_labels)
print(f'Test set — toxic: {cnt.get(1, cnt.get("toxic", "?"))},  non-toxic: {cnt.get(0, cnt.get("non-toxic", "?"))}')

Test set — toxic: 1000,  non-toxic: 1000


## 4. Load model & feature extractor (PhoWhisper-base)

In [8]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

MODEL_NAME = 'vinai/PhoWhisper-base'

processor = WhisperProcessor.from_pretrained(MODEL_NAME)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.config.dropout = 0.1
model.config.attention_dropout = 0.1

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens    = []

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {device}')

model.to(device);

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 657, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/vinai/PhoWhisper-base/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner


Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 68, in get_conversion_pr_reference
    pr = previous_pr(api, model_id, pr_title, token=token)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 14, in previous_pr
    for discussion in get_repo_discussions(repo_id=model_id, token=token):
    

generation_config.json: 0.00B [00:00, ?B/s]

Device: cuda


## 5. Tiền xử lý dữ liệu

In [9]:
!pip install num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 3.6 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=66f389c731c8dbf5267fbf19ceac9927156655bd65ee0b965e3ca213c71937a6
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [10]:
import re
from num2words import num2words  # pip install num2words nếu cần

def normalize_text(text: str) -> str:
    """Lowercase, remove punctuation, convert numbers to words."""
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    # Chuyển số thành chữ
    def replace_number(m):
        try:
            return num2words(int(m.group()), lang='vi')
        except Exception:
            return m.group()
    text = re.sub(r'\d+', replace_number, text)
    # Xóa dấu câu
    text = re.sub(r'[^\w\s]', '', text, flags=re.UNICODE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def prepare_dataset(batch):
    audio  = batch[AUDIO_COL]
    inputs = processor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    )
    batch['input_features'] = inputs.input_features[0]
    batch['labels'] = processor.tokenizer(
        normalize_text(batch[TEXT_COL])
    ).input_ids
    return batch


print('Đang tiền xử lý dữ liệu (có thể mất vài phút)...')
dataset = raw_dataset.map(
    prepare_dataset,
    remove_columns=raw_dataset['train'].column_names,
    # num_proc=1
)
print(' Tiền xử lý hoàn tất')
print(dataset)

Đang tiền xử lý dữ liệu (có thể mất vài phút)...


Map:   0%|          | 0/8641 [00:00<?, ? examples/s]

Map:   0%|          | 0/2161 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

 Tiền xử lý hoàn tất
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 8641
    })
    validation: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 2161
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 2000
    })
})


## 6. Data Collator

In [11]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [
            {'input_features': f['input_features']} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors='pt'
        )

        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch   = self.processor.tokenizer.pad(
            label_features, return_tensors='pt'
        )
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)
print('Data collator sẵn sàng')

Data collator sẵn sàng


## 7. Đánh giá WER trước fine-tuning (baseline)

In [12]:
import evaluate
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

wer_metric = evaluate.load('wer')

def compute_wer_on_test_batched(model, raw_test, processor, desc='Evaluating', batch_size=8):
    """Tính WER trên test set, sử dụng DataLoader để chạy đa mẫu (batching) giúp tăng tốc độ."""
    model.eval()
    all_preds, all_refs, all_labels = [], [], []

    # 1. Tạo hàm Data Collator để gộp nhiều mẫu thành 1 lô (batch)
    def collate_fn(batch):
        # Lấy mảng audio và thông tin text/label
        audios = [item[AUDIO_COL]['array'] for item in batch]
        sampling_rates = [item[AUDIO_COL]['sampling_rate'] for item in batch]
        refs = [normalize_text(item[TEXT_COL]) for item in batch]
        labels = [item.get(LABEL_COL, -1) for item in batch]

        # Xử lý 1 mẻ audio cùng lúc (Tự động padding)
        inputs = processor(
            audios,
            sampling_rate=sampling_rates[0], 
            return_tensors='pt'
        )
        return inputs.input_features, refs, labels

    # 2. Khởi tạo DataLoader
    # num_workers=2 giúp CPU đọc file audio ngầm trong lúc GPU đang generate
    dataloader = DataLoader(raw_test, batch_size=batch_size, collate_fn=collate_fn, num_workers=2)

    # Khai báo token ép buộc ngôn ngữ tiếng Việt để khắc phục ảo giác
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="vi", task="transcribe")

    # 3. Vòng lặp Evaluate theo lô
    for input_features, refs, labels in tqdm(dataloader, desc=desc):
        # Đưa input_features lên cùng thiết bị với model
        input_features = input_features.to(model.device)

        with torch.no_grad():
            # Sinh text cho cả 1 lô cùng lúc với các tham số giới hạn độ dài và ép ngôn ngữ
            pred_ids = model.generate(
                input_features, 
                forced_decoder_ids=forced_decoder_ids,
                max_new_tokens=255, # Ngăn chặn sinh từ vô hạn
                max_length=None,
                use_cache=True
            )
        
        # Decode cả 1 lô cùng lúc
        preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
        preds = [normalize_text(p) for p in preds]

        all_preds.extend(preds)
        all_refs.extend(refs)
        all_labels.extend(labels)

    # 4. Tính toán Metrics
    toxic_val = 1 
    toxic_idx    = [i for i, l in enumerate(all_labels) if l == toxic_val]
    nontoxic_idx = [i for i, l in enumerate(all_labels) if l != toxic_val and l != -1]

    def wer_subset(indices):
        if not indices:
            return float('nan')
        return wer_metric.compute(
            predictions=[all_preds[i] for i in indices],
            references=[all_refs[i]   for i in indices]
        )

    wer_all    = wer_metric.compute(predictions=all_preds, references=all_refs)
    wer_toxic  = wer_subset(toxic_idx)
    wer_nontox = wer_subset(nontoxic_idx)

    return {'toxic': wer_toxic, 'non_toxic': wer_nontox, 'all': wer_all, 'preds': all_preds}


print('=== WER BASELINE (w/o VITOSA fine-tuning) ===')
baseline_wer = compute_wer_on_test_batched(
    model, 
    raw_dataset['test'], 
    processor, 
    desc='Baseline WER',
    batch_size=16 
)

print(f"  Toxic    WER: {baseline_wer['toxic']:.3f}")
print(f"  Non-toxic WER: {baseline_wer['non_toxic']:.3f}")
print(f"  All WER:       {baseline_wer['all']:.3f}")

=== WER BASELINE (w/o VITOSA fine-tuning) ===


Baseline WER:   0%|          | 0/125 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

  Toxic    WER: 0.588
  Non-toxic WER: 0.166
  All WER:       0.382


## 8. Cấu hình Training 

In [13]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

# ---- Siêu tham số từ ----
training_args = Seq2SeqTrainingArguments(
    output_dir                 = './phowhisper-vitosa',
    num_train_epochs           = 10,        
    eval_strategy              = "epoch",     # Đã cập nhật đúng chuẩn mới
    per_device_train_batch_size= 4,           # Chỉnh lại thành 4 cho Kaggle T4x2 (4x2 = 8 batch size gốc)
    per_device_eval_batch_size = 8,
    optim                      = 'adamw_torch',  # AdamW
    learning_rate              = 5e-5,       
    warmup_ratio               = 0.1,         
    weight_decay               = 0.05,       
    gradient_checkpointing     = True,
    fp16                       = True,
    save_strategy              = 'epoch',
    load_best_model_at_end     = True,
    metric_for_best_model      = 'wer',
    greater_is_better          = False,
    predict_with_generate      = True,
    generation_max_length      = 225,
    logging_steps              = 50,
    report_to                  = 'none',
    dataloader_num_workers     = 2,
    push_to_hub                = False,
)
print('Training args sẵn sàng')
print(training_args)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training args sẵn sàng
Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOC

In [14]:
def compute_metrics(pred):
    pred_ids   = pred.predictions
    label_ids  = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str  = [normalize_text(p) for p in pred_str]
    label_str = [normalize_text(l) for l in label_str]

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}


trainer = Seq2SeqTrainer(
    model          = model,
    args           = training_args,
    train_dataset  = dataset['train'],
    eval_dataset   = dataset['validation'],   # dùng validation để chọn checkpoint tốt nhất
    data_collator  = data_collator,
    compute_metrics= compute_metrics,
    processing_class = processor.feature_extractor  # <-- ĐỔI TÊN Ở ĐÂY (từ tokenizer thành processing_class)
)
print('Trainer sẵn sàng')

Trainer sẵn sàng


## 9. Fine-tuning

In [15]:
print('Bắt đầu fine-tuning PhoWhisper-base trên VITOSA ...')
train_result = trainer.train()
print('\nFine-tuning hoàn tất!')
print(train_result.metrics)

Bắt đầu fine-tuning PhoWhisper-base trên VITOSA ...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Wer
1,1.666039,0.812172,0.465236
2,1.154613,0.736635,0.393697
3,0.695285,0.744633,0.371645
4,0.416913,0.795289,0.359017
5,0.213187,0.843436,0.358086
6,0.110073,0.877730,0.330782
7,0.058554,0.909274,0.327895
8,0.024927,0.939681,0.311076
9,0.012670,0.945641,0.304725
10,0.008087,0.957355,0.303142


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning hoàn tất!
{'train_runtime': 12318.3832, 'train_samples_per_second': 7.015, 'train_steps_per_second': 0.878, 'total_flos': 8.9087866085376e+18, 'train_loss': 0.49569924909631813, 'epoch': 10.0}


## 10. Đánh giá trên tập TEST (sau fine-tuning)

In [16]:
# Lưu model
trainer.save_model('./phowhisper-vitosa-best')
processor.save_pretrained('./phowhisper-vitosa-best')
print('Đã lưu model tốt nhất')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu model tốt nhất


In [17]:
# Dùng best model để đánh giá trên test set VITOSA
print('=== WER SAU FINE-TUNING VITOSA (test set) ===')
finetuned_wer = compute_wer_on_test_batched(
    model=trainer.model, 
    raw_test=raw_dataset['test'], 
    processor=processor, 
    desc='Fine-tuned WER',
    batch_size=16  # Dùng batch_size=16 giống với baseline để tăng tốc
)

print(f"  Toxic     WER: {finetuned_wer['toxic']:.3f}")
print(f"  Non-toxic WER: {finetuned_wer['non_toxic']:.3f}")
print(f"  All WER:       {finetuned_wer['all']:.3f}")

print('\n=== SO SÁNH BASELINE vs FINE-TUNED  ===')
print(f"  Toxic     WER: {baseline_wer['toxic']:.3f} → {finetuned_wer['toxic']:.3f}  "
      f"(↓{baseline_wer['toxic'] - finetuned_wer['toxic']:.3f})")
print(f"  Non-toxic WER: {baseline_wer['non_toxic']:.3f} → {finetuned_wer['non_toxic']:.3f}  "
      f"(↓{baseline_wer['non_toxic'] - finetuned_wer['non_toxic']:.3f})")
print(f"  All WER:       {baseline_wer['all']:.3f} → {finetuned_wer['all']:.3f}  "
      f"(↓{baseline_wer['all'] - finetuned_wer['all']:.3f})")

import json
with open('asr_predict.json', 'w', encoding='utf-8') as f:
    json.dump(finetuned_wer['preds'], f, ensure_ascii=False, indent=4)
print('\nĐã lưu kết quả dự đoán của mô hình ASR (fine-tuned) ra asr_predict.json')

=== WER SAU FINE-TUNING VITOSA (test set) ===


Fine-tuned WER:   0%|          | 0/125 [00:00<?, ?it/s]

  Toxic     WER: 0.360
  Non-toxic WER: 0.235
  All WER:       0.299

=== SO SÁNH BASELINE vs FINE-TUNED  ===
  Toxic     WER: 0.588 → 0.360  (↓0.228)
  Non-toxic WER: 0.166 → 0.235  (↓-0.069)
  All WER:       0.382 → 0.299  (↓0.083)

Đã lưu kết quả dự đoán của mô hình ASR (fine-tuned) ra asr_predict.json


## 11. Ví dụ định tính 

Hiển thị kết quả nhận diện câu toxic trước & sau fine-tuning.

In [18]:
# Lấy 3 ví dụ toxic từ test set để hiển thị định tính
toxic_samples = [s for s in raw_dataset['test'] if s.get(LABEL_COL) == 1][:3]

from transformers import WhisperForConditionalGeneration, WhisperProcessor
baseline_model = WhisperForConditionalGeneration.from_pretrained('vinai/PhoWhisper-base').to(device)

print(f"{'Ground Truth':40s} | {'Baseline':40s} | {'Fine-tuned':40s}")
print('-' * 125)

for s in toxic_samples:
    ref   = normalize_text(s[TEXT_COL])
    audio = s[AUDIO_COL]

    inp = processor(
        audio['array'], sampling_rate=audio['sampling_rate'], return_tensors='pt'
    ).input_features.to(device)

    with torch.no_grad():
        base_ids = baseline_model.generate(inp)
        fine_ids = trainer.model.generate(inp)

    base_pred = normalize_text(processor.decode(base_ids[0], skip_special_tokens=True))
    fine_pred = normalize_text(processor.decode(fine_ids[0], skip_special_tokens=True))

    print(f"{ref[:40]:40s} | {base_pred[:40]:40s} | {fine_pred[:40]:40s}")

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 657, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/vinai/PhoWhisper-base/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py

Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Ground Truth                             | Baseline                                 | Fine-tuned                              
-----------------------------------------------------------------------------------------------------------------------------
lúc nào cũng phải trong tư thế là phải n | lúc nào không phải trong tư thế là phải  | lúc nào mà phải trong tư thế là phải như
đụ má cái não mày cất dô cái hồ bơi hay  | đội máy cho não mình cất giữa hội vây dự | đụ má cho não mình cứt dô ở hậu vô gì vậ
đụ má thầy bắn đụ má muốn ỉa ra quần mấy | đổ mát thành bán đổ máy bốn tỷ lý ở quản | đụ má thầy bắn đụ má muốn ỉa như nào quầ
